In [9]:
import torch
import torch.nn as nn

device = "cuda"
torch.manual_seed(42)

In [3]:
from functools import partial
import torch.nn.functional as F

class ResidualUnit(nn.Module):
    def __init__(self, n_in:int, n_out:int, stride=1):
        super().__init__()
        DefaultConv2d = partial(
            nn.Conv2d, kernel_size=3, stride=1, padding=1, bias=False)
        self.main_layers = nn.Sequential(
            DefaultConv2d(n_in, n_out, stride=stride),
            nn.BatchNorm2d(n_out),
            nn.ReLU(),
            DefaultConv2d(n_out, n_out),
            nn.BatchNorm2d(n_out)
        )
        if stride > 1:
            self.skip_connection = nn.Sequential(
                DefaultConv2d(n_in, n_out, kernel_size=1, stride=stride, padding=0),
                nn.BatchNorm2d(n_out)
            )
        else:
            self.skip_connection = nn.Identity()
    
    def forward(self, inputs):
        return F.relu(self.main_layers(inputs) + self.skip_connection(inputs))

In [4]:
class ResNet32(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        layers = [
            nn.Conv2d(3, 64, 7, 2, 3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(3,2,1)
        ]
        prev_filters = 64
        for filters in [64] * 3 + [128] * 4 + [256] * 6 + [512] * 3:
            stride = 1 if filters == prev_filters else 2
            layers.append(ResidualUnit(prev_filters, filters, stride))
            prev_filters = filters
        layers += [
            nn.AdaptiveAvgPool2d(output_size=1),
            nn.Flatten(),
            nn.LazyLinear(10)
        ]
        self.resnet = nn.Sequential(*layers)
    
    def forward(self, inputs):
        return self.resnet(inputs)

In [7]:
import torchvision

weights = torchvision.models.ConvNeXt_Base_Weights.IMAGENET1K_V1
model = torchvision.models.convnext_base(weights=weights).to(device)

In [ ]:
# transformers = weights.transforms()
# preprocessed_images = transformers(sample_images_permuted)

In [16]:
from torch.utils.data import DataLoader
import torchvision.transforms.v2 as T

transforms = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=30),
    T.RandomResizedCrop(size=(224,224), scale=(0.8, 1.0)),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

DefaultFlowers102 = partial(torchvision.datasets.Flowers102, root="datasets",
                            transform=weights.transforms(), download=True)
train_set = DefaultFlowers102(split="train", transform=T)
valid_set = DefaultFlowers102(split="val")
test_set = DefaultFlowers102(split="test")

In [ ]:
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=32)
test_loader = DataLoader(test_set, batch_size=32)

In [13]:
[name for name, child in model.named_children()]

['features', 'avgpool', 'classifier']

In [14]:
model.classifier

Sequential(
  (0): LayerNorm2d((1024,), eps=1e-06, elementwise_affine=True, bias=True)
  (1): Flatten(start_dim=1, end_dim=-1)
  (2): Linear(in_features=1024, out_features=1000, bias=True)
)

In [15]:
n_classes = 102
model.classifier[2] = nn.Linear(1024, n_classes).to(device)

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

# todo : train model

In [ ]:
class FlowerLocator(nn.Module):
    def __init__(self, base_model) -> None:
        super().__init__()
        self.base_model = base_model
        self.localization_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(base_model.classifier[2].in_features, 4)
        )
    
    def forward(self, X):
        features = self.base_model.features(X)
        pool = self.base_model.avgpool(features)
        logits = self.base_model.classifier(pool)
        bbox = self.localization_head(pool)
        return logits, bbox

torch.manual_seed(42)
locator_model = FlowerLocator(model).to(device)

# y_pred_logits, y_pred_bbox = locator_model(some model)

In [ ]:
import torchvision.tv_tensors

bbox = torchvision.tv_tensors.BoundingBoxes(
    [[377,199,248,262]],
    format="CXCYWH",
    canvas_size=(500, 754)
)

# torchvision.utils.draw_bounding_boxes
